# 20 · Load: NYS DOL long-term occupational projections → `staging`

**Source card:** `docs/source-cards/nysdol_projections.md` (§6 issues, §7 staging table). Explored in: notebook 02. Decision: D1 (2022-32 defect).

**What this notebook does:** reads the four NYC projection editions and the official 2022-32 regional workbook from `data/raw/`, applies the card's issues list, and writes two staging tables:

| Table | One row = | Expected rows |
|---|---|---|
| `stg_nysdol__occupation_projections` | edition × SOC code (NYC) | 2,575 |
| `stg_nysdol__projections_xlsx` | SOC code (NYC, 2022-32) | 657 |

**Rules for every load notebook:** read raw files as text; every filter prints rows in → out; no value becomes NULL without being counted; the key is asserted unique; values are kept as published (fixes such as the D1 fill happen in `core`, not here).

If the database isn't set up yet, the tables are checked and saved to `data/interim/staging/*.parquet`, and the load is skipped. Re-run after the database exists.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "20-load-nysdol-projections.ipynb"
conn = connect()

FILES = pd.concat([raw_files("nysdol_projections"), raw_files("nysdol_projections_archive")], ignore_index=True)
FILES = FILES[~FILES.file.duplicated()]
display(FILES[["key", "file", "downloaded_utc", "bytes"]])

,key,file,downloaded_utc,bytes
0,nysdol_projections,jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv,2026-10-12T17:03:42+00:00,677541
1,nysdol_projections_archive,jobs/nysdol_projections_archive/pqm4-9qqb_2016-2026_wayback2021032...,2026-10-12T17:03:46+00:00,660955
2,nysdol_projections_archive,jobs/nysdol_projections_archive/pqm4-9qqb_2018-2028_wayback2022113...,2026-10-12T17:03:48+00:00,659060
3,nysdol_projections_archive,jobs/nysdol_projections_archive/pqm4-9qqb_2020-2030_wayback2024110...,2026-10-12T17:03:53+00:00,647901


## 1. Read the four CSVs as text and keep NYC

Issue 1: each file mixes two editions (regions = N, statewide = N+2), so the edition comes from `Period` on each row. Issue 3: the two Wayback files put an apostrophe before every SOC code. Issue 4: one exact duplicate row (`39-7000`, 2018-28).

In [2]:
NUM = {"Base Year": "base_employment", "Projected Year": "projected_employment", "Change": "employment_change",
       "Percent": "pct_change", "Total Annual Openings": "annual_openings_avg",
       "Growth Annual Openings": "growth_openings_avg", "Replacement Annual Openings": "separation_openings_avg"}

parts = []
for f, path in zip(FILES.file, FILES.path):
    d = pd.read_csv(path, dtype=str, keep_default_na=False)
    nyc = d[d.Area == "New York City"].assign(source_file=f)
    log(f"{f.split('/')[-1][:48]}: keep Area = New York City", len(d), len(nyc))
    parts.append(nyc)
p = pd.concat(parts, ignore_index=True)
rows_in = len(p)

p["SOC"] = p.SOC.str.lstrip("'")                                      # issue 3
before = len(p)
p = p.drop_duplicates(subset=[c for c in p.columns if c != "source_file"])  # issue 4
log("drop exact duplicate rows", before, len(p))
p["edition"] = p.Period.str.replace(" ", "")                           # issue 2
p[["base_year", "projected_year"]] = p.edition.str.split("-", expand=True).astype(int)
print(p.groupby("edition").size().rename("rows").to_frame().T)

  pqm4-9qqb_2026-10-12.csv: keep Area = New York City: 6,758 -> 657 rows (-6,101)
  pqm4-9qqb_2016-2026_wayback20210324183534.csv: keep Area = New York City: 6,626 -> 655 rows (-5,971)
  pqm4-9qqb_2018-2028_wayback20221130141045.csv: keep Area = New York City: 6,617 -> 638 rows (-5,979)
  pqm4-9qqb_2020-2030_wayback20241104204940.csv: keep Area = New York City: 6,596 -> 626 rows (-5,970)
  drop exact duplicate rows: 2,576 -> 2,575 rows (-1)
edition  2016-2026  2018-2028  2020-2030  2022-2032
rows           655        637        626        657


## 2. Numbers: text → integers, with nothing coerced silently

Every numeric column is parsed with `errors="raise"`, so a stray footnote mark or blank would stop the notebook instead of turning into NULL. Issue 6: `Base Year` / `Projected Year` hold employment, so they're renamed.

In [3]:
for src, dst in NUM.items():
    blanks = (p[src].str.strip() == "").sum()
    assert blanks == 0, f"{src}: {blanks} blank cells"
    p[dst] = pd.to_numeric(p[src], errors="raise")
INT = [c for c in NUM.values() if c != "pct_change"]
for c in INT:
    assert (p[c] % 1 == 0).all(), f"{c} has fractions"
    p[c] = p[c].astype("Int64")
p["pct_change"] = p["pct_change"].astype("float64")
p[list(NUM.values())].describe().T[["min", "50%", "max"]]

,min,50%,max
base_employment,250.0,3870.0,4939450.0
projected_employment,250.0,4310.0,5704650.0
employment_change,-23480.0,400.0,1138030.0
pct_change,-31.8,13.3,135.8
annual_openings_avg,1.0,430.0,653913.0
growth_openings_avg,-2348.0,40.0,113803.0
separation_openings_avg,0.0,400.0,580720.0


## 3. SOC level, leaf flag and SOC edition (issues 7–8)

All levels are stacked, and the hierarchy is ragged (e.g. `31-1120` has no detail below it), so `is_leaf` is computed per edition: a row is a leaf when no more detailed code under it appears in the same edition. Only leaves can be summed. SOC 2010 for 2016-26 and 2018-28, SOC 2018 after (nb 02 §4).

In [4]:
def soc_level(s: str) -> str:
    if s == "00-0000":
        return "total"
    t = s[3:]
    return "major" if t == "0000" else "minor" if t.endswith("00") else "broad" if t.endswith("0") else "detailed"

PREFIX = {"total": 0, "major": 2, "minor": 4, "broad": 5, "detailed": 6}
RANK = {"total": 0, "major": 1, "minor": 2, "broad": 3, "detailed": 4}
p["soc_level"] = p.SOC.map(soc_level)

def leaf_flags(g: pd.DataFrame) -> pd.Series:
    codes = [(c.replace("-", ""), soc_level(c)) for c in g.SOC]
    return g.apply(lambda r: not any(c.startswith(r.SOC.replace("-", "")[:PREFIX[r.soc_level]]) and RANK[l] > RANK[r.soc_level]
                                     for c, l in codes), axis=1)

p["is_leaf"] = p.groupby("edition", group_keys=False)[["SOC", "soc_level"]].apply(leaf_flags).astype(bool)
p["soc_edition"] = p.edition.map({"2016-2026": "2010", "2018-2028": "2010", "2020-2030": "2018", "2022-2032": "2018"})
assert p.soc_edition.notna().all()
display(pd.crosstab(p.edition, p.soc_level, margins=True))

# Leaves vs the published total, per edition (nb 02 §7: 98.6 / 98.0 / 97.1 / 92.2%)
tot = p[p.SOC == "00-0000"].set_index("edition").base_employment
cov = (p[p.is_leaf].groupby("edition").base_employment.sum() / tot).rename("leaf share of total")
display(cov.map("{:.1%}".format).to_frame().T)

soc_level,broad,detailed,major,minor,total,All
edition,,,,,,
2016-2026,0,542,22,90,1,655
2018-2028,3,523,20,90,1,637
2020-2030,10,499,22,94,1,626
2022-2032,7,533,22,94,1,657
All,20,2097,86,368,4,2575


edition,2016-2026,2018-2028,2020-2030,2022-2032
leaf share of total,98.6%,98.0%,97.1%,92.2%


## 4. Quality flag (issues 10–11, decision D1)

- **`publisher_defect`**: 2022-32 rows whose code is a BLS projection aggregate (not in SOC 2018) and has 0 separation openings. Every collapsed code has this signature (Software Developers `15-1256` = 600 jobs, 0 separations), while the 5 real BLS aggregates (`21-1018`, `25-2052`, `25-9045`, `51-2028`, `53-1047`) have normal separations. Core replaces these rows with the D1 fill (`imputed`).
- **`covid_base`**: every 2020-30 row (base year = 2020 COVID trough). Kept, but left out of "fastest growing" by default.
- **`published`**: everything else.

In [5]:
soc2018 = pd.read_excel(raw_files("bls_soc_structure").path.iloc[-1], header=None, dtype=str)
SOC2018 = set(soc2018.stack().str.strip()[lambda s: s.str.match(r"^\d\d-\d{4}$")])
print(f"SOC 2018 structure: {len(SOC2018):,} codes")

cur = p.edition == "2022-2032"
not_soc = ~p.SOC.isin(SOC2018) & (p.SOC != "00-0000")
defect = cur & not_soc & (p.separation_openings_avg == 0)
p["quality_flag"] = "published"
p.loc[p.edition == "2020-2030", "quality_flag"] = "covid_base"
p.loc[defect, "quality_flag"] = "publisher_defect"
display(p[cur & not_soc][["SOC", "Occupation Title", "base_employment", "separation_openings_avg", "quality_flag"]])
assert defect.sum() == 11 and {"15-1256", "13-2098", "15-1257"} <= set(p[defect].SOC)
p.groupby(["edition", "quality_flag"]).size().unstack(fill_value=0)

SOC 2018 structure: 1,447 codes


,SOC,Occupation Title,base_employment,separation_openings_avg,quality_flag
31,11-9198,"Personal Service Managers; Entertainment & Recreation Managers, Ex...",16900,0,publisher_defect
60,13-2098,"Financial and Investment Analysts, Financial Risk Specialists, and...",590,0,publisher_defect
71,15-1256,Software Developers and Software Quality Assurance Analysts and Te...,600,0,publisher_defect
72,15-1257,Web Developers and Digital Interface Designers,730,0,publisher_defect
116,19-3031,"Clinical, Counseling, and School Psychologists",1390,0,publisher_defect
134,21-1018,"Substance Abuse, Behavioral Disorder, and Mental Health Counselors",9200,870,published
200,25-2052,"Special Education Teachers, Kindergarten and Elementary School",7640,560,published
208,25-3097,"Teachers and Instructors, All Other, Except Substitute Teachers",1690,0,publisher_defect
218,25-9045,"Teaching Assistants, Except Postsecondary",42390,5320,published
288,29-1228,"Physicians, All Other; and Ophthalmologists, Except Pediatric",900,0,publisher_defect


quality_flag,covid_base,published,publisher_defect
edition,,,
2016-2026,0,655,0
2018-2028,0,637,0
2020-2030,626,0,0
2022-2032,0,646,11


## 5. Stage `stg_nysdol__occupation_projections`

In [6]:
stg = (p.rename(columns={"SOC": "soc_code", "Occupation Title": "soc_title"})
        [["edition", "base_year", "projected_year", "soc_code", "soc_edition", "soc_title", "soc_level", "is_leaf",
          *NUM.values(), "quality_flag", "source_file"]]
        .sort_values(["edition", "soc_code"]).reset_index(drop=True))
stg["soc_title"] = stg.soc_title.str.strip()
log("NYC rows in -> staged", rows_in, len(stg))
summary = [stage(stg, "stg_nysdol__occupation_projections", ["edition", "soc_code"], expected_rows=2_575,
                 source_key="nysdol_projections", notebook=NB, conn=conn,
                 comment="NYC long-term occupational projections, 4 editions, all SOC levels stacked (sum is_leaf rows only).")]

  NYC rows in -> staged: 2,576 -> 2,575 rows (-1)
  stg_nysdol__occupation_projections: 2,575 rows x 17 cols, key ['edition', 'soc_code'] unique | local data/interim/staging/stg_nysdol__occupation_projections.parquet | DB: 2575


## 6. The official 2022-32 regional workbook (NYC sheet)

Same numbers as the CSV (tie-out below), plus columns the CSV lacks. Kept here: NYS DOL Employment Prospects, the NYS 2020-30 growth rate and the 2026 wages (as a cross-check of `stg_nysdol__occupation_wages`). Entry education, experience and training are not staged from here: BLS EP (nb 22) is the source of record, and the workbook's blank cells can't be told apart (blank = "None" for detailed jobs, blank = not applicable for groups).

Layout: 6 header rows, then one row per code with `Summary Level` 1–4, then footnotes. Rows are kept where `Summary Level` is 1–4.

In [7]:
xf = raw_files("nysdol_projections_xlsx").iloc[-1]
xr = pd.read_excel(xf.path, sheet_name="New York City", header=None, skiprows=6, dtype=object).iloc[:, :22]
xr.columns = ["summary_level", "soc_code", "soc_title", "base_employment", "projected_employment", "employment_change",
              "pct_change", "annual_openings_avg", "growth_openings_avg", "separation_openings_avg", "exit_openings_avg",
              "transfer_openings_avg", "wage_mean_2026", "wage_median_2026", "wage_entry_2026", "wage_experienced_2026",
              "entry_education", "entry_experience", "entry_training", "attainment_most_common", "employment_prospects",
              "nys_growth_rate_2020_2030"]
x = xr[xr.summary_level.isin([1, 2, 3, 4])].copy()
log("NYC sheet: keep Summary Level 1-4 (drop header text and footnotes)", len(xr), len(x))

# Tie-out: every row and value identical to the CSV's 2022-32 NYC slice
t = x.merge(stg[stg.edition == "2022-2032"], on="soc_code", how="outer", suffixes=("_x", "_csv"), indicator=True)
assert (t._merge == "both").all(), t._merge.value_counts()
for c in ["base_employment", "projected_employment", "annual_openings_avg", "separation_openings_avg"]:
    assert (t[c + "_x"].astype(int) == t[c + "_csv"]).all(), c
print(f"Tie-out: {len(t)} codes, base / projected / openings / separations identical to the CSV")

# Blank handling, counted: " " (a space) and empty cells are both 'no value'
WAGES = ["wage_mean_2026", "wage_median_2026", "wage_entry_2026", "wage_experienced_2026"]
for c in WAGES + ["nys_growth_rate_2020_2030"]:
    assert x[c].map(lambda v: v is None or isinstance(v, (int, float))).all(), f"{c} has text"
    x[c] = x[c].astype("float64")
x["employment_prospects"] = x.employment_prospects.astype(str).str.strip().replace({"": None, "nan": None})
blank = x[WAGES + ["employment_prospects"]].isna().groupby(x.summary_level).sum()
print("Blank cells by summary level (1 = total ... 4 = detailed):"); display(blank)

x["summary_level"] = x.summary_level.astype("Int64")
x["soc_title"] = x.soc_title.astype(str).str.strip()
x["quality_flag"] = x.soc_code.map(stg[stg.edition == "2022-2032"].set_index("soc_code").quality_flag)
x["prospects_usable"] = x.quality_flag != "publisher_defect"
x["source_file"] = xf.file
display(x[~x.prospects_usable][["soc_code", "soc_title", "base_employment", "employment_prospects"]])

  NYC sheet: keep Summary Level 1-4 (drop header text and footnotes): 680 -> 657 rows (-23)
Tie-out: 657 codes, base / projected / openings / separations identical to the CSV
Blank cells by summary level (1 = total ... 4 = detailed):


,wage_mean_2026,wage_median_2026,wage_entry_2026,wage_experienced_2026,employment_prospects
summary_level,,,,,
1,0,0,0,0,1
2,0,0,0,0,22
3,94,94,94,94,94
4,32,32,32,33,0


,soc_code,soc_title,base_employment,employment_prospects
32,11-9198,"Personal Service Managers; Entertainment & Recreation Managers, Ex...",16900,Unfavorable
61,13-2098,"Financial and Investment Analysts, Financial Risk Specialists, and...",590,Unfavorable
72,15-1256,Software Developers and Software Quality Assurance Analysts and Te...,600,Unfavorable
73,15-1257,Web Developers and Digital Interface Designers,730,Unfavorable
117,19-3031,"Clinical, Counseling, and School Psychologists",1390,Unfavorable
209,25-3097,"Teachers and Instructors, All Other, Except Substitute Teachers",1690,Unfavorable
289,29-1228,"Physicians, All Other; and Ophthalmologists, Except Pediatric",900,Unfavorable
291,29-1298,Acupuncturists and Healthcare Diagnosing or Treating Practitioners...,490,Unfavorable
382,39-1098,First-Line Supervisors of Personal Service & Entertainment and Rec...,1860,Unfavorable
408,39-9098,"Crematory Operators and Personal Care and Service Workers, All Other",1450,Unfavorable


Employment Prospects is wrong for the 11 defect codes. NYS DOL computed it from the collapsed numbers, so Software Developers come out "Unfavorable". The column is staged as published, with `prospects_usable = false` on those rows; the dashboard never shows it for them.

In [8]:
stg_x = x[["soc_code", "soc_title", "summary_level", *WAGES, "employment_prospects", "nys_growth_rate_2020_2030",
           "quality_flag", "prospects_usable", "source_file"]].sort_values("soc_code").reset_index(drop=True)
stg_x.insert(0, "edition", "2022-2032")
summary.append(stage(stg_x, "stg_nysdol__projections_xlsx", ["soc_code"], expected_rows=657,
                     source_key="nysdol_projections_xlsx", notebook=NB, conn=conn,
                     comment="NYC sheet of the official 2022-32 regional workbook: Employment Prospects, 2026 wages, NYS rate."))

  stg_nysdol__projections_xlsx: 657 rows x 13 cols, key ['soc_code'] unique | local data/interim/staging/stg_nysdol__projections_xlsx.parquet | DB: 657


## 7. Summary

In [9]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_nysdol__occupation_projections,2575,17,edition + soc_code,nysdol_projections,20-load-nysdol-projections.ipynb,2575,2026-10-14T21:09:27+00:00
1,stg_nysdol__projections_xlsx,657,13,soc_code,nysdol_projections_xlsx,20-load-nysdol-projections.ipynb,657,2026-10-14T21:09:27+00:00
